In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("GPT_MODEL", "your-gpt-model"),
    base_url=os.getenv("PORTKEY_BASE_URL", "your-base-url"),
    api_key=os.getenv("PORTKEY_API_KEY", "your-api-key"),
)
print(llm.invoke("Say hi in five words.").content)

Hello there, hope you're well.


In [3]:
#  Add memory - using input_key and output_key for better control
agent = create_agent(
    llm,
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver(),  # this is the "buffer memory"
)
config = {"configurable": {"thread_id": "chat-1"}}


def chat(text: str) -> str:
    out = agent.invoke({"messages": [HumanMessage(text)]}, config)
    return out["messages"][-1].text

In [4]:
name = input("Enter your name: ")
print("\n Greet:\n", chat(f"My name is {name}. Please remember it and greet me."))

print("\n Asking if it remembers your name...")
print("\n Response:\n", chat("Do you remember my name?"))


 Greet:
 Hello, Savvoulis! Nice to meet you. I’ll remember your name for this chat session. How can I help you today?

 Asking if it remembers your name...

 Response:
 Yes, I remember your name is Savvoulis. I’ll remember it for this chat session. How can I help you today?


In [5]:
for m in agent.get_state(config).values["messages"]:
    print(f"{m.type.upper():6} : {m.text}")

HUMAN  : My name is Savvoulis. Please remember it and greet me.
AI     : Hello, Savvoulis! Nice to meet you. I’ll remember your name for this chat session. How can I help you today?
HUMAN  : Do you remember my name?
AI     : Yes, I remember your name is Savvoulis. I’ll remember it for this chat session. How can I help you today?
